In [40]:
import pennylane as qp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [39]:
df = pd.read_csv("bank_marketing_qos_encoded.csv")

In [38]:
rng = np.random.default_rng(42)

yes_indices = df.index[df["y"] == "yes"].to_numpy()
no_indices = df.index[df["y"] == "no"].to_numpy()

rng.shuffle(yes_indices)
rng.shuffle(no_indices)

yes_split = int(0.8 * len(yes_indices))
no_split = int(0.8 * len(no_indices))

train_indices = np.concatenate([
    yes_indices[:yes_split],
    no_indices[:no_split]
])

test_indices = np.concatenate([
    yes_indices[yes_split:],
    no_indices[no_split:]
])

train_df = df.loc[train_indices]
test_df = (
    df.loc[test_indices]
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

print(test_df["y"].head(20))

print("Train size:", len(train_df))
print("Test size:", len(test_df))

0      no
1     yes
2      no
3      no
4      no
5      no
6      no
7     yes
8      no
9      no
10     no
11     no
12     no
13     no
14     no
15     no
16     no
17    yes
18     no
19     no
Name: y, dtype: object
Train size: 36168
Test size: 9043


In [41]:
num_addresses = 240
quantum_dimension = 256
num_qubits = 8

alpha = 1.0

train_yes = train_df[train_df["y"] == "yes"]
train_no = train_df[train_df["y"] == "no"]

counts_yes_train = np.bincount(
    train_yes["qos_address"].to_numpy(),
    minlength=num_addresses
)

counts_no_train = np.bincount(
    train_no["qos_address"].to_numpy(),
    minlength=num_addresses
)

p_yes_train = (
    counts_yes_train + alpha
) / (
    counts_yes_train.sum()
    + alpha * num_addresses
)

p_no_train = (
    counts_no_train + alpha
) / (
    counts_no_train.sum()
    + alpha * num_addresses
)

In [42]:
p_yes_padded = np.zeros(quantum_dimension)
p_no_padded = np.zeros(quantum_dimension)

p_yes_padded[:num_addresses] = p_yes_train
p_no_padded[:num_addresses] = p_no_train

In [43]:
def decode_qos_address(address):
    x = int(address)

    loan_code = x % 2
    x //= 2

    housing_code = x % 2
    x //= 2

    job_code = x % len(job_categories)
    x //= len(job_categories)

    age_code = x

    return {
        "age_bin": age_categories[age_code],
        "job": job_categories[job_code],
        "housing": binary_categories[housing_code],
        "loan": binary_categories[loan_code]
    }

age_categories = [
    "under_30",
    "30_39",
    "40_49",
    "50_59",
    "60_plus"
]

job_categories = [
    "admin.",
    "blue-collar",
    "entrepreneur",
    "housemaid",
    "management",
    "retired",
    "self-employed",
    "services",
    "student",
    "technician",
    "unemployed",
    "unknown"
]

binary_categories = ["no", "yes"]
training_address_counts = np.bincount(
    train_df["qos_address"].to_numpy(),
    minlength=num_addresses
)

reference_address = int(
    np.argmax(training_address_counts)
)

print("Reference address:", reference_address)
print(
    "Number of training customers at reference:",
    training_address_counts[reference_address]
)

Reference address: 54
Number of training customers at reference: 2059


In [44]:
print(decode_qos_address(reference_address))

{'age_bin': '30_39', 'job': 'blue-collar', 'housing': 'yes', 'loan': 'no'}


In [45]:
def make_customer_state(address, reference_address, dimension=256):
    
    state = np.zeros(dimension, dtype=complex)

    if address == reference_address:
        state[address] = 1.0
    else:
        state[reference_address] = 1 / np.sqrt(2)
        state[address] = 1 / np.sqrt(2)

    return state

In [46]:
example_address = int(
    test_df["qos_address"].iloc[0]
)

example_state = make_customer_state(
    example_address,
    reference_address
)

print("Example address:", example_address)
print("Norm:", np.sum(np.abs(example_state) ** 2))
print("Nonzero amplitudes:", np.count_nonzero(example_state))

Example address: 39
Norm: 0.9999999999999998
Nonzero amplitudes: 2


In [47]:
t_classifier = 20.0

yes_diagonal = np.exp(
    1j * t_classifier * p_yes_padded
)

no_diagonal = np.exp(
    1j * t_classifier * p_no_padded
)

In [48]:
wires = list(range(num_qubits))

dev = qp.device(
    "default.qubit",
    wires=num_qubits
)

In [49]:
@qp.qnode(dev)
def customer_oracle_circuit(
    customer_state,
    oracle_diagonal
):
    
    qp.StatePrep(
        customer_state,
        wires=wires
    )

    qp.DiagonalQubitUnitary(
        oracle_diagonal,
        wires=wires
    )

    return qp.state()

In [50]:
yes_evolved = customer_oracle_circuit(
    example_state,
    yes_diagonal
)

print(yes_evolved.shape)

(256,)


In [51]:
def quantum_similarity(
    address,
    oracle_diagonal,
    reference_address
):
    
    customer_state = make_customer_state(
        address,
        reference_address
    )

    evolved_state = customer_oracle_circuit(
        customer_state,
        oracle_diagonal
    )

    overlap = np.vdot(
        customer_state,
        evolved_state
    )

    return float(
        np.abs(overlap) ** 2
    )

In [52]:
score_yes = quantum_similarity(
    example_address,
    yes_diagonal,
    reference_address
)

score_no = quantum_similarity(
    example_address,
    no_diagonal,
    reference_address
)

print("Example address:", example_address)
print("Yes score:", score_yes)
print("No score:", score_no)

Example address: 39
Yes score: 0.9214608796882091
No score: 0.7023065533989632


In [53]:
def predict_quantum_customer(address):
    
    score_yes = quantum_similarity(
        address,
        yes_diagonal,
        reference_address
    )

    score_no = quantum_similarity(
        address,
        no_diagonal,
        reference_address
    )

    if score_yes > score_no:
        return "yes"
    else:
        return "no"

In [54]:
for i in range(10):
    
    row = test_df.iloc[i]

    address = int(row["qos_address"])
    true_label = row["y"]

    score_yes = quantum_similarity(
        address,
        yes_diagonal,
        reference_address
    )

    score_no = quantum_similarity(
        address,
        no_diagonal,
        reference_address
    )

    prediction = (
        "yes"
        if score_yes > score_no
        else "no"
    )

    print(
        f"Address={address:3d} | "
        f"True={true_label:3s} | "
        f"Yes score={score_yes:.6f} | "
        f"No score={score_no:.6f} | "
        f"Pred={prediction}"
    )

Address= 39 | True=no  | Yes score=0.921461 | No score=0.702307 | Pred=yes
Address= 52 | True=yes | Yes score=0.974993 | No score=0.808845 | Pred=yes
Address=144 | True=no  | Yes score=0.970632 | No score=0.745352 | Pred=yes
Address=126 | True=no  | Yes score=0.954601 | No score=0.797478 | Pred=yes
Address=112 | True=no  | Yes score=0.999280 | No score=0.862359 | Pred=yes
Address=134 | True=no  | Yes score=0.966738 | No score=0.840259 | Pred=yes
Address=182 | True=no  | Yes score=0.940681 | No score=0.773503 | Pred=yes
Address=204 | True=yes | Yes score=0.938550 | No score=0.698320 | Pred=yes
Address= 59 | True=no  | Yes score=0.923851 | No score=0.698035 | Pred=yes
Address= 54 | True=no  | Yes score=1.000000 | No score=1.000000 | Pred=no


In [55]:
def quantum_similarity_fast(
    address,
    p_class,
    reference_address,
    t
):
    phase_difference = (
        p_class[address]
        - p_class[reference_address]
    )

    return np.cos(
        t * phase_difference / 2
    ) ** 2

In [56]:
def predict_quantum_customer_fast(address, t=20.0):

    score_yes = quantum_similarity_fast(
        address,
        p_yes_train,
        reference_address,
        t
    )

    score_no = quantum_similarity_fast(
        address,
        p_no_train,
        reference_address,
        t
    )

    if score_yes > score_no:
        return "yes"
    else:
        return "no"

In [57]:
true_labels = test_df["y"].to_numpy()

quantum_predictions = np.array([
    predict_quantum_customer_fast(
        int(address),
        t=20.0
    )
    for address in test_df["qos_address"].to_numpy()
])

yes_mask = true_labels == "yes"
no_mask = true_labels == "no"

yes_accuracy_quantum = np.mean(
    quantum_predictions[yes_mask] == "yes"
)

no_accuracy_quantum = np.mean(
    quantum_predictions[no_mask] == "no"
)

balanced_accuracy_quantum = (
    yes_accuracy_quantum
    + no_accuracy_quantum
) / 2

overall_accuracy_quantum = np.mean(
    quantum_predictions == true_labels
)

print("Yes-class accuracy:", yes_accuracy_quantum)
print("No-class accuracy:", no_accuracy_quantum)
print(
    "Balanced accuracy:",
    balanced_accuracy_quantum
)
print(
    "Overall accuracy:",
    overall_accuracy_quantum
)

Yes-class accuracy: 0.9054820415879017
No-class accuracy: 0.09768315591734503
Balanced accuracy: 0.5015825987526233
Overall accuracy: 0.1921928563529802


In [58]:
print("Reference address:", reference_address)
print("p_yes(ref):", p_yes_train[reference_address])
print("p_no(ref):", p_no_train[reference_address])

Reference address: 54
p_yes(ref): 0.029299932900917022
p_no(ref): 0.05998073157845666
